# IFN647 - Assignment 2
## Information Retrieval and Ranking on the RCV1 Subset (Dataset101 - Dataset150)

**Group:** *<Group Number>*  
**Members:** *<Student 1, Student 2, Student 3>*  
**Contributions:** equal (33.3% each)

This notebook implements the three required ranking models and the full evaluation pipeline:

| Model | Description | Equation in spec |
|-------|-------------|------------------|
| **Baseline 1** | BM25 IR model | Eq. (1) |
| **Baseline 2** | Jelinek-Mercer (JM) smoothing language model | Eq. (2) |
| **Model_C** | BM25 + Pseudo Relevance Feedback (PRF) re-ranking | Lecture/W10 PRM material |

The code reuses the data structures and pre-processing pipeline introduced in the Week 10 workshop solution (`coll.py`, `df.py`, `IRM_bm25.py`, `PRM_training_bm25_wk10.py`, `PRM_test_eval_bm25_wk10.py`). Only the modifications required by the A2 specification have been made (e.g. `k2 = 500`, `log10`, the JM smoothing formula and the PRF re-ranking).

## 0. Setup

Allowed packages used in this assignment: `nltk`, `sklearn`, `pandas`, `numpy`, `matplotlib` and the Python standard library. The `stemming` package (Porter2) is used because it is the stemmer used in the Week 10 workshop solution (`coll.py`); per the assignment outline, "you can re-use or update your assignment 1 code, the workshop solutions or review question solutions", so its use is allowed.

No other third-party libraries are required: the paired t-test in Section 10 is implemented from scratch with `numpy` and the standard library, so `scipy` is **not** needed.

In [25]:
import os
import re
import math
import glob
import string
import statistics
from math import log2, sqrt, erf

import numpy as np
import pandas as pd
from stemming.porter2 import stem

# Project paths (this notebook lives in Assignments/A2/)
BASE_DIR = os.path.abspath(os.curdir)
TOPICS_FILE  = os.path.join(BASE_DIR, 'Topics.txt')
DOC_COLL_DIR = os.path.join(BASE_DIR, 'Doc_Collection')
JUDGE_DIR    = os.path.join(BASE_DIR, 'Relevant_Judgements')
OUT_DIR      = os.path.join(BASE_DIR, 'ModelOutputs')
os.makedirs(OUT_DIR, exist_ok=True)

print('BASE_DIR     :', BASE_DIR)
print('TOPICS_FILE  :', TOPICS_FILE)
print('DOC_COLL_DIR :', DOC_COLL_DIR)
print('JUDGE_DIR    :', JUDGE_DIR)
print('OUT_DIR      :', OUT_DIR)

BASE_DIR     : /Users/niceenb/Documents/647/Assignment2
TOPICS_FILE  : /Users/niceenb/Documents/647/Assignment2/Topics.txt
DOC_COLL_DIR : /Users/niceenb/Documents/647/Assignment2/Doc_Collection
JUDGE_DIR    : /Users/niceenb/Documents/647/Assignment2/Relevant_Judgements
OUT_DIR      : /Users/niceenb/Documents/647/Assignment2/ModelOutputs


In [26]:
# Stop word list (taken from the Week 10 workshop file common-english-words.txt).
STOP_WORDS = (
    'a,able,about,across,after,all,almost,also,am,among,an,and,any,are,as,at,'
    'be,because,been,but,by,can,cannot,could,dear,did,do,does,either,else,ever,'
    'every,for,from,get,got,had,has,have,he,her,hers,him,his,how,however,i,if,'
    'in,into,is,it,its,just,least,let,like,likely,may,me,might,most,must,my,'
    'neither,no,nor,not,of,off,often,on,only,or,other,our,own,rather,said,say,'
    'says,she,should,since,so,some,than,that,the,their,them,then,there,these,'
    'they,this,tis,to,too,twas,us,wants,was,we,were,what,when,where,which,'
    'while,who,whom,why,will,with,would,yet,you,your'
).split(',')
print(f'Loaded {len(STOP_WORDS)} stop words.')

Loaded 119 stop words.


## 1. Document data structures (`BowDoc`, `BowColl`)

These are copied verbatim from the Week 10 workshop solution `coll.py`. A `BowDoc` is a bag-of-words representation of a single document (`docid`, `terms` dict and `doc_len`). A `BowColl` is the collection of `BowDoc` objects keyed by their docid.

In [27]:
class BowDoc:
    """Bag-of-words representation of a document."""

    def __init__(self, docid):
        self.docid = docid
        self.terms = {}
        self.doc_len = 0

    def add_term(self, term):
        try:
            self.terms[term] += 1
        except KeyError:
            self.terms[term] = 1

    def get_term_count(self, term):
        try:
            return self.terms[term]
        except KeyError:
            return 0

    def get_term_freq_dict(self):
        return self.terms

    def get_term_list(self):
        return sorted(self.terms.keys())

    def get_docid(self):
        return self.docid

    def get_doc_len(self):
        return self.doc_len

    def set_doc_len(self, doc_len):
        self.doc_len = doc_len


class BowColl:
    """Collection of BOW documents."""

    def __init__(self):
        self.docs = {}

    def add_doc(self, doc):
        self.docs[doc.get_docid()] = doc

    def get_doc(self, docid):
        return self.docs[docid]

    def get_docs(self):
        return self.docs

    def get_num_docs(self):
        return len(self.docs)

## 2. XML parser (`parse_rcv_coll`) and document frequency (`calc_df`)

The pre-processing is identical to the W10 workshop solution:
- only text between `<text>` and `</text>` is kept;
- digits and punctuation are removed;
- words are lower-cased;
- Porter2 stem is applied;
- terms shorter than 3 characters and stop words are dropped;
- `doc_len` is the **raw** word count (before stop-word/short-term removal), matching `coll.py`.

The only modification w.r.t. the workshop version is that we **do not** mutate the current working directory (the workshop's `os.chdir(inputpath)` would break a loop over 50 datasets). We use `glob.glob(os.path.join(inputpath, '*.xml'))` instead.

In [28]:
def queryParser(query, stop_words):
    """Parse query string into {term: frequency} dict."""
    query_terms = {}
    line = (query.translate(str.maketrans('', '', string.digits)).translate(
        str.maketrans(string.punctuation, ' ' * len(string.punctuation))))
    for term in line.split():
        term = stem(term.lower())
        if len(term) > 2 and term not in stop_words:
            try:
                query_terms[term] += 1
            except KeyError:
                query_terms[term] = 1
    return query_terms

In [29]:
def parse_rcv_coll(inputpath, stop_words):
    """Parse all RCV1 XML files under `inputpath` into a BowColl.

    Faithful to the Week 10 workshop solution; only the chdir was removed
    so the function can be called repeatedly on different folders."""
    coll = BowColl()
    for file_ in glob.glob(os.path.join(inputpath, '*.xml')):
        curr_doc = None
        start_end = False
        word_count = 0
        for line in open(file_, encoding='utf-8', errors='ignore'):
            line = line.strip()
            if start_end is False:
                if line.startswith('<newsitem '):
                    for part in line.split():
                        if part.startswith('itemid='):
                            docid = part.split('=')[1].split('"')[1]
                            curr_doc = BowDoc(docid)
                            break
                    continue
                if line.startswith('<text>'):
                    start_end = True
            elif line.startswith('</text>'):
                break
            elif curr_doc is not None:
                line = line.replace('<p>', '').replace('</p>', '')
                line = line.translate(str.maketrans('', '', string.digits)) \
                           .translate(str.maketrans(string.punctuation,
                                                    ' ' * len(string.punctuation)))
                for term in line.split():
                    word_count += 1
                    term = stem(term.lower())
                    if len(term) > 2 and term not in stop_words:
                        curr_doc.add_term(term)
        if curr_doc is not None:
            curr_doc.set_doc_len(word_count)
            coll.add_doc(curr_doc)
    return coll


def calc_df(coll):
    """Document frequency of each term in the collection (from df.py)."""
    df_ = {}
    for _id, doc in coll.get_docs().items():
        for term in doc.get_term_list():
            try:
                df_[term] += 1
            except KeyError:
                df_[term] = 1
    return df_

## 3. Topics parser

`Topics.txt` contains 50 topic blocks of the form `<num> Number: R### ... <title> ...`. We extract the topic id and the title (which is used as the query).

In [30]:
def parse_topics(path):
    """Return an ordered dict {topic_id: title_string}."""
    text = open(path, encoding='utf-8', errors='ignore').read()
    topics = {}
    # Split into per-topic chunks at each <num> Number: marker
    blocks = re.split(r'<num>\s*Number:\s*', text)
    for block in blocks[1:]:
        m_id    = re.match(r'(R\d+)', block)
        m_title = re.search(r'<title>\s*(.+)', block)
        if m_id and m_title:
            tid   = m_id.group(1)
            title = m_title.group(1).strip()
            topics[tid] = title
    return topics

topics = parse_topics(TOPICS_FILE)
print(f'Parsed {len(topics)} topics.')
for tid in list(topics)[:5]:
    print(f'  {tid}: {topics[tid]!r}')

Parsed 50 topics.
  R101: 'Economic espionage'
  R102: 'Convicts, repeat offenders'
  R103: 'Ferry Boat sinkings'
  R104: 'Rescue of kidnapped children'
  R105: 'Sport Utility Vehicles U.S.'


## 4. Baseline 1 - BM25 (Equation 1)

$$BM25(D,Q_i)=\sum_{t\in Q_i}\log_{10}\!\Big(1+\frac{N-n_t+0.5}{n_t+0.5}\Big)\cdot\frac{(k_1+1)f_t}{K+f_t}\cdot\frac{(k_2+1)qf_t}{k_2+qf_t}$$

with $k_1=1.2$, $k_2=500$, $b=0.75$ and $K=k_1\cdot((1-b)+b\cdot dl/avdl)$.

**Meaning of variables:**
- **$N$** - number of documents in `Dataset_i`.
- **$n_t$** - document frequency of term $t$ (how many documents in `Dataset_i` contain $t$).
- **$f_t$** - term frequency of $t$ inside document $D$.
- **$qf_t$** - frequency of $t$ inside the query $Q_i$.
- **$dl$** - length of document $D$ (number of word occurrences before stop-word removal).
- **$avdl$** - average document length in `Dataset_i`.

The skeleton is the workshop `IRM_bm25.bm25` function; only `k2=500`, `log10` and the spec's formulation `log10(1 + (N-n+0.5)/(n+0.5))` were substituted.

In [31]:
def avg_doc_len(coll):
    """Average document length over a BowColl (from IRM_bm25.py)."""
    tot_dl = 0
    for _id, doc in coll.get_docs().items():
        tot_dl = tot_dl + doc.get_doc_len()
    return tot_dl / coll.get_num_docs()


def bm25(coll, q, df):
    """BM25 score for every document in `coll` given query string `q`.

    Implements Equation (1) of the A2 specification.
    Parameters k1=1.2, k2=500, b=0.75 as stated in the spec.
    """
    k1, k2, b = 1.2, 500, 0.75
    bm25s = {}
    avg_dl   = avg_doc_len(coll)
    no_docs  = coll.get_num_docs()
    # Pre-compute query term frequencies (with the same stemming / lower-casing as docs).
    qfs = queryParser(q, STOP_WORDS)
    for _id, doc in coll.get_docs().items():
        K = k1 * ((1 - b) + b * (doc.get_doc_len() / float(avg_dl)))
        bm25_ = 0.0
        for qt, qf in qfs.items():
            if qt in df:
                n = df[qt]
                f = doc.get_term_count(qt)
                if f == 0:
                    continue
                bm25_ += math.log10(1.0 + (no_docs - n + 0.5) / (n + 0.5)) \
                       * ((k1 + 1) * f) / (K + f) \
                       * ((k2 + 1) * qf) / (k2 + qf)
        bm25s[doc.get_docid()] = bm25_
    return bm25s

## 5. Baseline 2 - Jelinek-Mercer language model (Equation 2)

$$JM(D,Q_i)=\sum_{j=1}^{n}\log_{10}\!\Big((1-\lambda)\frac{f_{q_j,D}}{|D|}+\lambda\frac{c_{q_j}}{|Dataset_i|}\Big),\qquad \lambda=0.3$$

**Meaning of variables:**
- **$n$** - number of query terms in $Q_i$ (after pre-processing).
- **$f_{q_j,D}$** - frequency of (stemmed) query term $q_j$ in document $D$.
- **$c_{q_j}$** - collection frequency of $q_j$ in dataset $Dataset_i$ ($\sum_{D\in Dataset_i} f_{q_j,D}$).
- **$|D|$** - number of (stemmed/filtered) word occurrences in $D$, i.e. $\sum_t f_{t,D}$.
- **$|Dataset_i|$** - total number of (stemmed/filtered) word occurrences in $Dataset_i$ ($\sum_D |D|$).
- **$\lambda=0.3$** - smoothing parameter that mixes document evidence with collection evidence.

**Why filtered counts for $|D|$ and $|Dataset_i|$.** $f_{q_j,D}$ is read from the stemmed/filtered term-frequency dictionary, so for the probability ratio $f_{q_j,D}/|D|$ to be internally consistent the denominator must live in the same (stemmed/filtered) vocabulary space. This is also what is needed to reproduce the JM example scores shown in the spec.

In [32]:
def index_docs(inputpath,stop_words):
    Index = {}    # initialize the index
    os.chdir(inputpath)
    for file_ in glob.glob("*.xml"):
        start_end = False
        for line in open(file_):
            line = line.strip()
            if(start_end == False):
                if line.startswith("<newsitem "):
                    for part in line.split():
                        if part.startswith("itemid="):
                            docid = part.split("=")[1].split("\"")[1]
                            break
                if line.startswith("<text>"):
                    start_end = True
            elif line.startswith("</text>"):
                break
            else:
                line = line.replace("<p>", "").replace("</p>", "")
                line = line.translate(str.maketrans('','', string.digits)).translate(str.maketrans(string.punctuation, ' '*len(string.punctuation)))
                for term in line.split():
                    term = stem(term.lower())
                    if len(term) > 2 and term not in stop_words:
                        try:
                            try:
                                Index[term][docid] += 1
                            except KeyError:
                                Index[term][docid]=1
                        except KeyError:
                            Index[term] = {docid:1}
    return Index

In [33]:
from math import log10
def likelihood_JM(I_C, I_D, Q, lamda):  # index I_C ans I_D have the form of {term:{itemId:freq}}
    # calcualte query term frequency in documents indexed by I_D
    L={}    # L is the selected inverted list
    R={}    # R is a directionary of docId:score
    D_len={} # D_len is a directionary of docId:length
    for list in I_D.items():
        for id in list[1].items():
            R[id[0]]=0.0       # get all document IDs and initialize as 0.0 for log accumulation
            D_len[id[0]]=0.5 # initialize a small non-zero value as it will be used as denominator
        if (list[0] in Q):     # select inverted lists based on the query
                L[list[0]]= I_D[list[0]]
    for q_term in Q.items(): # L may not include all query terms
        if not(q_term[0] in L):
            L[q_term[0]]={}
    for list in I_D.items():
        for id in list[1].items(): # Count term occurrences in documents
            D_len[id[0]]= D_len[id[0]] + id[1]
    # calculate query term frequency in I_C
    CF={}
    L_C={}
    for list in I_C.items():
        if (list[0] in Q):     # select inverted lists based on Q in I_C
                L_C[list[0]]= I_C[list[0]]
                CF[list[0]] = 0 # assign 0 to each query term
    for (term, doc) in L_C.items():
        for id in doc.items():
            CF[term] = CF[term] + id[1]
    C_len = 0
    for list in I_C.items():
        for id in list[1].items(): # Count term occurrences in documents
            C_len = C_len + id[1]
    # using the equation
    for (d, sd) in R.items():
        for (term, f) in L.items():
            #if not(d in f):
            #    f[d]=0
            tf = f.get(d, 0)
            cf = CF.get(term, 0)
            p = (1 - lamda) * (tf / D_len[d]) + lamda * (cf / C_len)
            if p > 0:
                sd += log10(p)
            #sd = sd + log10((1-lamda)*f[d]/D_len[d]+(lamda*CF[term]/C_len)) # see page 12 in wk5 lecture notes
        R[d] = sd
    return R

## 6. Model_C - BM25 + Pseudo Relevance Feedback (PRF)

Model_C extends Baseline 1 with the **pseudo-relevance feedback** technique presented in the Week 10 workshop (`PRM_training_bm25_wk10.py`, `PRM_test_eval_bm25_wk10.py`).

**Algorithm.** For each topic $i$:
1. Compute initial BM25 scores `bm25_init` for every document in `Dataset_i`.
2. Treat the top-$K$ documents as a *pseudo-relevant* set (`ben[id] = 1`) and the remaining as non-relevant (`ben[id] = 0`).
3. Compute the **w5** weights for every term in those pseudo-relevant documents using the workshop formula
   $$w5(t)=\log\!\Big(\frac{(r_t+0.5)/(R-r_t+0.5)}{(n_t-r_t+0.5)/(N-n_t-R+r_t+0.5)}\Big)$$
   and apply the feature-selection rule `r > mean(w5) + theta`.
4. For every document compute a feature score `fs(D) = sum_{t in features, t in D} features[t]` (the workshop's `BM25Testing` function).
5. Min-max normalise `bm25_init` and `fs` per topic and combine them:
   $$Score_{C}(D,Q_i)=\alpha\,\widetilde{BM25}(D,Q_i)+(1-\alpha)\,\widetilde{fs}(D)$$

**Parameters of Model_C** (tuned in Section 9):
- $K$ - size of the pseudo-relevant set (number of top BM25 documents kept).
- $\theta$ - feature-selection threshold above the mean of w5 weights.
- $\alpha\in[0,1]$ - weight of the original BM25 score in the final ranking.

**Difference from the baselines.**
- Baseline 1 (BM25) ranks purely on the query terms.
- Baseline 2 (JM) ranks on a smoothed unigram language model with a fixed $\lambda$, no learning.
- Model_C uses BM25 only as a *first pass*, then **automatically expands the query** with the most discriminative terms in the top-$K$ documents and re-ranks. It is a generic, topic-agnostic procedure and exposes three tunable parameters.

In [34]:
def w5(coll, ben, theta):
    """w5 feature-selection function from PRM_training_bm25_wk10.py.

    `ben` is a dict {docid: 1 or 0} - here built from pseudo-relevance.
    Returns a dict {term: weight} for the selected features.
    """
    T = {}
    # r(t_k) = number of pseudo-relevant docs containing t_k
    for _id, doc in coll.get_docs().items():
        if ben.get(_id, 0) > 0:
            for term, _freq in doc.terms.items():
                try:
                    T[term] += 1
                except KeyError:
                    T[term] = 1
    # n(t_k) = document frequency of t_k in the whole collection
    ntk = {}
    for _id, doc in coll.get_docs().items():
        for term in doc.get_term_list():
            try:
                ntk[term] += 1
            except KeyError:
                ntk[term] = 1
    No_docs = coll.get_num_docs()
    R = sum(1 for v in ben.values() if v > 0)
    if R == 0:
        return {}
    for term, rtk in T.items():
        T[term] = ((rtk + 0.5) / (R - rtk + 0.5)) / \
                  ((ntk[term] - rtk + 0.5) / (No_docs - ntk[term] - R + rtk + 0.5))
    # Feature selection: keep terms with w5 > mean + theta
    if len(T) == 0:
        return {}
    meanW5 = sum(T.values()) / len(T)
    Features = {t: r for t, r in T.items() if r > meanW5 + theta}
    return Features


def feature_score(coll, features):
    """Sum of selected feature weights for every document (from BM25Testing)."""
    ranks = {}
    for _id, doc in coll.get_docs().items():
        s = 0.0
        terms = doc.get_term_list()
        for term in features.keys():
            if term in terms:
                s += features[term]
        ranks[_id] = s
    return ranks


def _min_max(d):
    """Min-max normalise a dict of {key: score} to [0, 1]."""
    if not d:
        return {}
    vs = list(d.values())
    lo, hi = min(vs), max(vs)
    rng = hi - lo
    if rng <= 0:
        return {k: 0.0 for k in d}
    return {k: (v - lo) / rng for k, v in d.items()}


def model_c(coll, q, df, K=10, theta=3.0, alpha=0.5):
    """Model_C: BM25 + Pseudo Relevance Feedback re-ranking.

    K     - number of top BM25 documents used as pseudo-relevant.
    theta - feature-selection threshold above mean(w5).
    alpha - weight of BM25 in the final combined score.
    """
    # 1. Initial BM25 ranking
    bm25_init = bm25(coll, q, df)

    # 2. Pseudo-relevance set: top-K BM25 documents = "1", rest = "0"
    sorted_ids = [d for d, _ in sorted(bm25_init.items(), key=lambda x: x[1], reverse=True)]
    ben = {d: (1 if i < K else 0) for i, d in enumerate(sorted_ids)}

    # 3. Expansion features via w5
    features = w5(coll, ben, theta)

    # 4. Feature score for every document
    fs = feature_score(coll, features)

    # 5. Normalise and combine
    n_bm = _min_max(bm25_init)
    n_fs = _min_max(fs)
    scores = {d: alpha * n_bm.get(d, 0.0) + (1 - alpha) * n_fs.get(d, 0.0)
              for d in bm25_init}
    return scores

## 7. Run the three models on all 50 datasets

Produces `Baseline1_R###_Ranking.dat`, `Baseline2_R###_Ranking.dat` and `ModelC_R###_Ranking.dat` in the `ModelOutputs/` folder for each topic. Each file is the ranked list `<docid> <score>` in descending score order, as required by the spec.

In [36]:
for topic_id, topic_title in sorted(topics.items()):
    try:
        dataset_folder = os.path.join(DOC_COLL_DIR, f'Dataset{topic_id[1:]}')

        _cwd = os.getcwd()
        Index_D = index_docs(dataset_folder, STOP_WORDS)
        os.chdir(_cwd)
        Index_C = Index_D

        # parse topic title into query term-frequency dict
        Q_dict = queryParser(topic_title, STOP_WORDS)

        # compute JM scores (lambda = 0.3 as required)
        jm_scores = likelihood_JM(Index_C, Index_D, Q_dict, 0.3)

        # sort and write output in the requested format
        ranked_jm = sorted(jm_scores.items(), key=lambda x: x[1], reverse=True)
        output_file_jm = os.path.join(OUT_DIR, f"Baseline2_{topic_id}_Ranking.dat")
        with open(output_file_jm, 'w', encoding='utf-8') as f2:
            f2.write(f"Query is the topic title = \"{topic_title}\"\n")
            f2.write("Doc_ID JM_Score\n")
            for docid, score in ranked_jm:
                f2.write(f"{docid} {score}\n")
        print(f"Baseline2_{topic_id}_Ranking.dat")
        print(f'Query is the topic title = "{topic_title}"')
        print('Doc_ID JM_Score')
        for docid, score in ranked_jm:
            print(f'{docid} {score}')
    except Exception as e:
        print(f'  JM ranking failed for {topic_id}: {e}')

Baseline2_R101_Ranking.dat
Query is the topic title = "Economic espionage"
Doc_ID JM_Score
46974 -3.29765692690395
46547 -3.29765692690395
62325 -4.1351055379194595
61329 -4.940335231962925
6146 -5.042058755984156
61780 -5.276485539423247
22170 -5.3047170465883795
22513 -5.397161692683204
82330 -5.594194774960731
39496 -5.70807123892863
82454 -6.188969260396155
18586 -6.188969260396155
27577 -6.188969260396155
30647 -6.188969260396155
26847 -6.188969260396155
82912 -6.188969260396155
83167 -6.188969260396155
80425 -6.188969260396155
81463 -6.188969260396155
80950 -6.188969260396155
77909 -6.188969260396155
63261 -6.188969260396155
26642 -6.188969260396155
Baseline2_R102_Ranking.dat
Query is the topic title = "Convicts, repeat offenders"
Doc_ID JM_Score
78836 -6.13262457295431
58476 -6.344224975897699
57914 -6.652203039330916
26061 -6.6800224093596245
76635 -6.7837582738951685
73038 -6.964733537266456
12769 -6.977340819171906
12767 -7.084117101168557
25096 -7.26967248004571
4358 -7.3762

In [47]:
def save_ranking(path, score_dict):
    """Write a ranking dict as `<docid> <score>` lines, sorted by score desc."""
    with open(path, 'w') as f:
        for docid, sc in sorted(score_dict.items(), key=lambda x: x[1], reverse=True):
            f.write(f'{docid} {sc}\n')


def run_all_models(topics, doc_coll_dir, out_dir,
                   stop_words, K=10, theta=3.0, alpha=0.5,
                   model_c_suffix='ModelC'):
    """Run the three models on every dataset and write the .dat files.

    Returns a dict {topic_id: {'b1': scores, 'b2': scores, 'mc': scores}}
    so it can be reused for evaluation without reading the files back.
    """
    all_scores = {}
    for tid, query in topics.items():
        dataset_path = os.path.join(doc_coll_dir, f'Dataset{tid[1:]}')
        coll  = parse_rcv_coll(dataset_path, stop_words)
        df_   = calc_df(coll)

        _cwd = os.getcwd()
        Index_D = index_docs(dataset_path, STOP_WORDS)
        os.chdir(_cwd)
        Index_C = Index_D

        # parse topic title into query term-frequency dict
        Q_dict = queryParser(query, STOP_WORDS)

        # compute JM scores (lambda = 0.3 as required)

        b1 = bm25(coll, query, df_)
        b2 = likelihood_JM(Index_C, Index_D, Q_dict, 0.3)
        mc = model_c(coll, query, df_, K=K, theta=theta, alpha=alpha)

        save_ranking(os.path.join(out_dir, f'Baseline1_{tid}_Ranking.dat'), b1)
        save_ranking(os.path.join(out_dir, f'Baseline2_{tid}_Ranking.dat'), b2)
        save_ranking(os.path.join(out_dir, f'{model_c_suffix}_{tid}_Ranking.dat'), mc)

        all_scores[tid] = {'b1': b1, 'b2': b2, 'mc': mc}
        print(f'  Processed {tid:>5} (|D|={coll.get_num_docs():3d})  query="{query}"')
    return all_scores

# Execute the full pipeline (generates 150 .dat files inside ModelOutputs/).
# The (K, theta, alpha) values below are the best-MAP combination found by
# the parameter validation grid in Section 9.
BEST_K, BEST_THETA, BEST_ALPHA = 5, 5.0, 0.5
all_scores = run_all_models(topics, DOC_COLL_DIR, OUT_DIR,
                            STOP_WORDS,
                            K=BEST_K, theta=BEST_THETA, alpha=BEST_ALPHA)
print('\nFinished. Output files:', len(os.listdir(OUT_DIR)))

  Processed  R101 (|D|= 23)  query="Economic espionage"
  Processed  R102 (|D|=199)  query="Convicts, repeat offenders"
  Processed  R103 (|D|= 64)  query="Ferry Boat sinkings"
  Processed  R104 (|D|=194)  query="Rescue of kidnapped children"
  Processed  R105 (|D|= 37)  query="Sport Utility Vehicles U.S."
  Processed  R106 (|D|= 44)  query="Government supported school vouchers"
  Processed  R107 (|D|= 61)  query="Tourism Great Britain"
  Processed  R108 (|D|= 53)  query="Harmful weight-loss drugs"
  Processed  R109 (|D|= 40)  query="Child custody cases"
  Processed  R110 (|D|= 91)  query="Terrorism Middle East tourism"
  Processed  R111 (|D|= 52)  query="Telemarketing practices U.S."
  Processed  R112 (|D|= 57)  query="School bus accidents"
  Processed  R113 (|D|= 68)  query="Ford foreign ventures"
  Processed  R114 (|D|= 25)  query="Effects of global warming"
  Processed  R115 (|D|= 46)  query="Indian casino laws"
  Processed  R116 (|D|= 46)  query="Archaeology discoveries"
  Process

### 7.1 Quick sanity check - top-10 of Baseline 1 for R101, R109, R135
Matches the example formatting used in the assignment specification (Task 3).

In [37]:
def print_top_n(score_dict, n=10):
    for docid, sc in sorted(score_dict.items(), key=lambda x: x[1], reverse=True)[:n]:
        print(f'{docid} {sc}')

for tid in ['R101', 'R109', 'R135']:
    print(f'\n{tid} (Doc_ID BM25_Score):')
    print_top_n(all_scores[tid]['b1'], 10)


R101 (Doc_ID BM25_Score):
46974 1.8878533347166444
46547 1.8878533347166444
62325 1.4153502848719022
6146 0.8552738202464255
61329 0.8018094461466141
22170 0.7463997606217948
61780 0.6405403133414163
22513 0.5456633261040381
82330 0.41055501722694937
39496 0.33280811450767495

R109 (Doc_ID BM25_Score):
16953 1.131236307194757
26073 1.0479843111751264
61540 1.038283542199531
4933 0.9661551929278844
16575 0.9043070702368079
64476 0.8730202244909762
67717 0.8497452282468656
78626 0.8418493584792329
34684 0.8112752050406455
23398 0.7921558872623953

R135 (Doc_ID BM25_Score):
57876 0.6744874940228573
71894 0.5196381472811176
42848 0.5172924578571716
2738 0.40620267332577525
79698 0.030066821554749767
78998 0.030066821554749767
65311 0.030065775903336065
65501 0.03006361368816656
52721 0.030049043784249163
80891 0.030028313847379404


In [38]:
for topic_id, topic_title in sorted(topics.items()):
    try:
        dataset_folder = os.path.join(DOC_COLL_DIR, f'Dataset{topic_id[1:]}')

        _cwd = os.getcwd()
        Index_D = index_docs(dataset_folder, STOP_WORDS)
        os.chdir(_cwd)
        Index_C = Index_D

        # parse topic title into query term-frequency dict
        Q_dict = queryParser(topic_title, STOP_WORDS)

        # compute JM scores (lambda = 0.3 as required)
        jm_scores = likelihood_JM(Index_C, Index_D, Q_dict, 0.3)

        # sort and write output in the requested format
        ranked_jm = sorted(jm_scores.items(), key=lambda x: x[1], reverse=True)
        output_file_jm = os.path.join(OUT_DIR, f"Baseline2_{topic_id}_Ranking.dat")
        with open(output_file_jm, 'w', encoding='utf-8') as f2:
            f2.write(f"Query is the topic title = \"{topic_title}\"\n")
            f2.write("Doc_ID JM_Score\n")
            for docid, score in ranked_jm:
                f2.write(f"{docid} {score}\n")
        print(f"Baseline2_{topic_id}_Ranking.dat")
        print(f'Query is the topic title = "{topic_title}"')
        print('Doc_ID JM_Score')
        for docid, score in ranked_jm:
            print(f'{docid} {score}')
    except Exception as e:
        print(f'  JM ranking failed for {topic_id}: {e}')

Baseline2_R101_Ranking.dat
Query is the topic title = "Economic espionage"
Doc_ID JM_Score
46974 -3.29765692690395
46547 -3.29765692690395
62325 -4.1351055379194595
61329 -4.940335231962925
6146 -5.042058755984156
61780 -5.276485539423247
22170 -5.3047170465883795
22513 -5.397161692683204
82330 -5.594194774960731
39496 -5.70807123892863
82454 -6.188969260396155
18586 -6.188969260396155
27577 -6.188969260396155
30647 -6.188969260396155
26847 -6.188969260396155
82912 -6.188969260396155
83167 -6.188969260396155
80425 -6.188969260396155
81463 -6.188969260396155
80950 -6.188969260396155
77909 -6.188969260396155
63261 -6.188969260396155
26642 -6.188969260396155
Baseline2_R102_Ranking.dat
Query is the topic title = "Convicts, repeat offenders"
Doc_ID JM_Score
78836 -6.13262457295431
58476 -6.344224975897699
57914 -6.652203039330916
26061 -6.6800224093596245
76635 -6.7837582738951685
73038 -6.964733537266456
12769 -6.977340819171906
12767 -7.084117101168557
25096 -7.26967248004571
4358 -7.3762

## 8. Evaluation: Average Precision (MAP), Precision@10, DCG@10

Definitions (from the A2 specification):
- **Average Precision** for a topic with $R$ relevant documents: $AP=\frac{1}{R}\sum_{i=1}^{|ranking|}\frac{rel_i\cdot \text{hits}_i}{i}$ where $rel_i\in\{0,1\}$ and $\text{hits}_i$ is the cumulative number of relevant docs up to position $i$. **MAP** is the mean of AP over the 50 topics.
- **Precision@10** = (# relevant in top 10) / 10.
- **DCG@10** = $rel_1 + \sum_{i=2}^{10} \frac{rel_i}{\log_2 i}$.

In [39]:
def load_judgments(path):
    """Load relevance judgements file `R### <docid> <0|1>`."""
    rels = {}
    for line in open(path):
        parts = line.strip().split()
        if len(parts) < 3:
            continue
        topic, docid, label = parts[-3], parts[-2], int(parts[-1])
        rels[docid] = label
    return rels


def ranking_from_scores(score_dict):
    """Return list of docids sorted by score descending."""
    return [d for d, _ in sorted(score_dict.items(), key=lambda x: x[1], reverse=True)]


def average_precision(ranking, rels):
    R = sum(1 for v in rels.values() if v == 1)
    if R == 0:
        return 0.0
    hits, ap = 0, 0.0
    for i, docid in enumerate(ranking, start=1):
        if rels.get(docid, 0) == 1:
            hits += 1
            ap += hits / i
    return ap / R


def precision_at_k(ranking, rels, k=10):
    top = ranking[:k]
    if not top:
        return 0.0
    return sum(1 for d in top if rels.get(d, 0) == 1) / k


def dcg_at_k(ranking, rels, k=10):
    top = ranking[:k]
    if not top:
        return 0.0
    dcg = (1 if rels.get(top[0], 0) == 1 else 0)
    for i in range(2, len(top) + 1):
        rel = 1 if rels.get(top[i - 1], 0) == 1 else 0
        dcg += rel / log2(i)
    return dcg

In [40]:
def evaluate_all(all_scores, topics, judge_dir):
    """Compute AP, P@10 and DCG@10 per topic for the three models.
    Returns a dict of three pd.DataFrame (one per metric)."""
    rows_ap, rows_p10, rows_dcg = [], [], []
    for tid in topics:
        rels = load_judgments(os.path.join(judge_dir, f'Dataset{tid[1:]}.txt'))
        r_b1 = ranking_from_scores(all_scores[tid]['b1'])
        r_b2 = ranking_from_scores(all_scores[tid]['b2'])
        r_mc = ranking_from_scores(all_scores[tid]['mc'])
        rows_ap.append({'Topic': tid,
                        'Baseline1': average_precision(r_b1, rels),
                        'Baseline2': average_precision(r_b2, rels),
                        'Model_C'  : average_precision(r_mc, rels)})
        rows_p10.append({'Topic': tid,
                         'Baseline1': precision_at_k(r_b1, rels, 10),
                         'Baseline2': precision_at_k(r_b2, rels, 10),
                         'Model_C'  : precision_at_k(r_mc, rels, 10)})
        rows_dcg.append({'Topic': tid,
                         'Baseline1': dcg_at_k(r_b1, rels, 10),
                         'Baseline2': dcg_at_k(r_b2, rels, 10),
                         'Model_C'  : dcg_at_k(r_mc, rels, 10)})
    df_ap  = pd.DataFrame(rows_ap)
    df_p10 = pd.DataFrame(rows_p10)
    df_dcg = pd.DataFrame(rows_dcg)
    return df_ap, df_p10, df_dcg


def add_average_row(df, label):
    avg = {'Topic': label,
           'Baseline1': df['Baseline1'].mean(),
           'Baseline2': df['Baseline2'].mean(),
           'Model_C'  : df['Model_C'].mean()}
    return pd.concat([df, pd.DataFrame([avg])], ignore_index=True)


df_ap, df_p10, df_dcg = evaluate_all(all_scores, topics, JUDGE_DIR)
table1 = add_average_row(df_ap.copy(),  'MAP')
table2 = add_average_row(df_p10.copy(), 'Average')
table3 = add_average_row(df_dcg.copy(), 'Average')

pd.options.display.float_format = '{:0.4f}'.format
print('Table 1. Average Precision per topic (and MAP)')
print(table1.to_string(index=False))

Table 1. Average Precision per topic (and MAP)
Topic  Baseline1  Baseline2  Model_C
 R101     0.6849     0.6869   0.7327
 R102     0.7925     0.7910   0.7882
 R103     0.3835     0.3523   0.4426
 R104     0.9153     0.8566   0.9209
 R105     0.7697     0.6501   0.7697
 R106     0.4252     0.4432   0.2577
 R107     0.3590     0.2701   0.5806
 R108     0.1285     0.1393   0.1242
 R109     0.6346     0.6918   0.5773
 R110     0.4867     0.3609   0.5200
 R111     0.1308     0.1308   0.1308
 R112     1.0000     1.0000   1.0000
 R113     0.6353     0.6075   0.6340
 R114     0.8850     0.9029   0.9250
 R115     0.3254     0.2583   0.5417
 R116     0.3011     0.2855   0.3011
 R117     0.2687     0.4833   0.2409
 R118     0.2944     0.2611   0.3778
 R119     0.1321     0.1343   0.1698
 R120     0.5235     0.5702   0.5062
 R121     0.6008     0.7334   0.6882
 R122     0.5528     0.5621   0.5528
 R123     0.4773     0.4606   0.2551
 R124     0.3905     0.5310   0.4646
 R125     0.7178     0.5489 

In [41]:
print('Table 2. precision@10 per topic (and average)')
print(table2.to_string(index=False))

Table 2. precision@10 per topic (and average)
  Topic  Baseline1  Baseline2  Model_C
   R101     0.5000     0.5000   0.5000
   R102     0.6000     0.6000   0.6000
   R103     0.5000     0.4000   0.5000
   R104     1.0000     0.9000   1.0000
   R105     0.7000     0.7000   0.7000
   R106     0.2000     0.2000   0.2000
   R107     0.2000     0.2000   0.2000
   R108     0.1000     0.1000   0.1000
   R109     0.6000     0.8000   0.6000
   R110     0.4000     0.4000   0.4000
   R111     0.0000     0.0000   0.0000
   R112     0.6000     0.6000   0.6000
   R113     0.6000     0.6000   0.6000
   R114     0.5000     0.5000   0.5000
   R115     0.2000     0.2000   0.2000
   R116     0.2000     0.1000   0.2000
   R117     0.2000     0.2000   0.2000
   R118     0.3000     0.2000   0.3000
   R119     0.0000     0.0000   0.1000
   R120     0.6000     0.5000   0.6000
   R121     0.6000     0.8000   0.6000
   R122     0.4000     0.5000   0.4000
   R123     0.1000     0.1000   0.1000
   R124     0.4000

In [42]:
print('Table 3. DCG@10 per topic (and average)')
print(table3.to_string(index=False))

Table 3. DCG@10 per topic (and average)
  Topic  Baseline1  Baseline2  Model_C
   R101     3.2474     3.2474   3.2653
   R102     3.4340     3.4340   3.1343
   R103     2.8740     2.4320   3.2431
   R104     5.2545     4.8676   5.2545
   R105     4.0228     2.7545   4.0228
   R106     1.3869     1.4307   1.0178
   R107     1.5000     1.0616   1.6309
   R108     0.3333     0.3333   0.3333
   R109     3.7073     4.5343   3.2073
   R110     1.9485     1.7797   2.3175
   R111     0.0000     0.0000   0.0000
   R112     3.9485     3.9485   3.9485
   R113     3.7474     3.2188   3.7474
   R114     3.3949     3.4178   3.4643
   R115     1.1309     0.9307   1.5000
   R116     1.3155     1.0000   1.3155
   R117     0.9320     1.3010   0.8010
   R118     1.1879     0.8175   1.6879
   R119     0.0000     0.0000   0.3010
   R120     3.1895     2.9178   3.1521
   R121     3.7104     4.5522   3.8740
   R122     2.0050     2.3060   2.0050
   R123     1.0000     1.0000   0.6309
   R124     2.1895     2

In [43]:
# Save the three tables to CSV for inclusion in the report appendices
table1.to_csv(os.path.join(BASE_DIR, 'Table1_AP.csv'),  index=False)
table2.to_csv(os.path.join(BASE_DIR, 'Table2_P10.csv'), index=False)
table3.to_csv(os.path.join(BASE_DIR, 'Table3_DCG10.csv'), index=False)

summary = pd.DataFrame({
    'Metric'   : ['MAP', 'P@10', 'DCG@10'],
    'Baseline1': [df_ap['Baseline1'].mean(), df_p10['Baseline1'].mean(), df_dcg['Baseline1'].mean()],
    'Baseline2': [df_ap['Baseline2'].mean(), df_p10['Baseline2'].mean(), df_dcg['Baseline2'].mean()],
    'Model_C'  : [df_ap['Model_C'].mean(),   df_p10['Model_C'].mean(),   df_dcg['Model_C'].mean()],
})
print('\nOverall summary (averaged over 50 topics):')
print(summary.to_string(index=False))


Overall summary (averaged over 50 topics):
Metric  Baseline1  Baseline2  Model_C
   MAP     0.4911     0.4718   0.4892
  P@10     0.3580     0.3580   0.3600
DCG@10     2.0900     2.0303   2.0639


## 9. Validation - parameter tuning for Model_C (Task 4(b))

Model_C has three parameters: $K$ (pseudo-relevant set size), $\theta$ (feature-selection threshold) and $\alpha$ (weight of the original BM25 score). We perform a small grid search **over the entire dataset collection** (50 datasets, as suggested in the spec hint) and pick the combination that maximises MAP, while monitoring P@10 and DCG@10 for stability.

The grid below is intentionally small to keep run time reasonable; widen it if you have more time.

In [44]:
def evaluate_model_c_grid(topics, doc_coll_dir, judge_dir, stop_words, grid):
    """For every (K, theta, alpha) in `grid`, run Model_C over all topics and
    return a DataFrame with MAP, P@10, DCG@10."""
    # Cache the parsed BowColl per topic to avoid re-parsing.
    coll_cache, df_cache, rels_cache = {}, {}, {}
    for tid in topics:
        c = parse_rcv_coll(os.path.join(doc_coll_dir, f'Dataset{tid[1:]}'), stop_words)
        coll_cache[tid] = c
        df_cache[tid]   = calc_df(c)
        rels_cache[tid] = load_judgments(os.path.join(judge_dir, f'Dataset{tid[1:]}.txt'))
    rows = []
    for (K, theta, alpha) in grid:
        ap_l, p10_l, dcg_l = [], [], []
        for tid, q in topics.items():
            scores  = model_c(coll_cache[tid], q, df_cache[tid], K=K, theta=theta, alpha=alpha)
            ranking = ranking_from_scores(scores)
            ap_l.append(average_precision(ranking, rels_cache[tid]))
            p10_l.append(precision_at_k(ranking, rels_cache[tid], 10))
            dcg_l.append(dcg_at_k(ranking, rels_cache[tid], 10))
        rows.append({'K': K, 'theta': theta, 'alpha': alpha,
                     'MAP'   : statistics.mean(ap_l),
                     'P@10'  : statistics.mean(p10_l),
                     'DCG@10': statistics.mean(dcg_l)})
        print(f'  K={K:2d} theta={theta:>3} alpha={alpha:.2f} -> MAP={rows[-1]["MAP"]:.4f}')
    return pd.DataFrame(rows)

grid = [(K, t, a)
        for K in (5, 10, 15)
        for t in (3.0, 5.0)
        for a in (0.3, 0.5, 0.7)]
validation = evaluate_model_c_grid(topics, DOC_COLL_DIR, JUDGE_DIR, STOP_WORDS, grid)
validation.to_csv(os.path.join(BASE_DIR, 'ModelC_Validation.csv'), index=False)
print('\nValidation results (sorted by MAP desc):')
print(validation.sort_values('MAP', ascending=False).to_string(index=False))

best = validation.sort_values('MAP', ascending=False).iloc[0]
print(f'\nSelected parameters: K={int(best["K"])} theta={best["theta"]} alpha={best["alpha"]}')

  K= 5 theta=3.0 alpha=0.30 -> MAP=0.4847
  K= 5 theta=3.0 alpha=0.50 -> MAP=0.4860
  K= 5 theta=3.0 alpha=0.70 -> MAP=0.4860
  K= 5 theta=5.0 alpha=0.30 -> MAP=0.4861
  K= 5 theta=5.0 alpha=0.50 -> MAP=0.4892
  K= 5 theta=5.0 alpha=0.70 -> MAP=0.4868
  K=10 theta=3.0 alpha=0.30 -> MAP=0.4679
  K=10 theta=3.0 alpha=0.50 -> MAP=0.4693
  K=10 theta=3.0 alpha=0.70 -> MAP=0.4730
  K=10 theta=5.0 alpha=0.30 -> MAP=0.4648
  K=10 theta=5.0 alpha=0.50 -> MAP=0.4658
  K=10 theta=5.0 alpha=0.70 -> MAP=0.4754
  K=15 theta=3.0 alpha=0.30 -> MAP=0.4469
  K=15 theta=3.0 alpha=0.50 -> MAP=0.4429
  K=15 theta=3.0 alpha=0.70 -> MAP=0.4647
  K=15 theta=5.0 alpha=0.30 -> MAP=0.4487
  K=15 theta=5.0 alpha=0.50 -> MAP=0.4465
  K=15 theta=5.0 alpha=0.70 -> MAP=0.4681

Validation results (sorted by MAP desc):
 K  theta  alpha    MAP   P@10  DCG@10
 5 5.0000 0.5000 0.4892 0.3600  2.0639
 5 5.0000 0.7000 0.4868 0.3640  2.0771
 5 5.0000 0.3000 0.4861 0.3640  2.0558
 5 3.0000 0.5000 0.4860 0.3600  2.0534
 5 3.00

## 10. Significance testing (Task 5) - paired t-test

We perform paired t-tests (50 topics) comparing Model_C against each baseline on **all three** effectiveness measures. The null hypothesis is that the means are equal; we use a significance threshold of $p = 0.05$.

In [45]:
def paired_t(metric_baseline, metric_model_c):
    """Paired two-sided t-test, implemented from scratch (numpy + math only).

    Steps (textbook formulae, no scipy used):
      d_i  = model_c_i - baseline_i             (per-topic differences)
      mean = average of d_i
      sd   = sample standard deviation of d_i   (ddof = 1)
      t    = mean / (sd / sqrt(n))              (paired t-statistic)
      p    = 2 * (1 - Phi(|t|))                 (two-sided p, normal approx.)

    With n = 50 (df = 49) the Student-t distribution is very close to the
    standard normal, so we use the normal CDF via math.erf for the p-value.
    Returns (t_stat, p_val).  If all differences are identical we return
    (0.0, 1.0) (no significant difference, well-defined edge case).
    """
    diffs = np.array(metric_model_c, dtype=float) - np.array(metric_baseline, dtype=float)
    n = len(diffs)
    if n < 2 or float(diffs.max() - diffs.min()) == 0.0:
        return 0.0, 1.0
    mean_d = float(diffs.mean())
    sd_d   = float(diffs.std(ddof=1))
    if sd_d == 0.0:
        return 0.0, 1.0
    t_stat = mean_d / (sd_d / sqrt(n))
    # Two-sided p-value via standard normal approximation (df = n - 1 = 49 here)
    p_val  = 2.0 * (1.0 - 0.5 * (1.0 + erf(abs(t_stat) / sqrt(2))))
    return t_stat, p_val

tests = []
for metric_name, df_metric in [('AP / MAP', df_ap), ('P@10', df_p10), ('DCG@10', df_dcg)]:
    for base in ['Baseline1', 'Baseline2']:
        t_stat, p_val = paired_t(df_metric[base].tolist(),
                                 df_metric['Model_C'].tolist())
        tests.append({'Metric'     : metric_name,
                      'Comparison' : f'Model_C vs {base}',
                      't-statistic': t_stat,
                      'p-value'    : p_val,
                      'Significant (p<0.05)': bool(p_val < 0.05)})

tests_df = pd.DataFrame(tests)
tests_df.to_csv(os.path.join(BASE_DIR, 'Significance_Tests.csv'), index=False)
print('Paired t-tests (Model_C vs each baseline) across 50 topics:')
print(tests_df.to_string(index=False))

Paired t-tests (Model_C vs each baseline) across 50 topics:
  Metric           Comparison  t-statistic  p-value  Significant (p<0.05)
AP / MAP Model_C vs Baseline1      -0.1349   0.8927                 False
AP / MAP Model_C vs Baseline2       0.9116   0.3620                 False
    P@10 Model_C vs Baseline1       0.4436   0.6573                 False
    P@10 Model_C vs Baseline2       0.1444   0.8852                 False
  DCG@10 Model_C vs Baseline1      -0.6470   0.5176                 False
  DCG@10 Model_C vs Baseline2       0.3437   0.7311                 False


**How to read the table.** A positive `t-statistic` with `p-value < 0.05` indicates that Model_C is *significantly* better than the corresponding baseline on that metric. A `p-value` above 0.05 means the observed improvement is not statistically significant; in that case the recommendation in the report should be honest about the lack of significance.

## 11. Top-10 results for every topic (Appendix material)

Writes three plain-text appendix files containing the top-10 ranked documents per topic, ready to paste into the final report.

In [46]:
def write_appendix(model_key, header_label, file_name):
    out_path = os.path.join(BASE_DIR, file_name)
    with open(out_path, 'w') as f:
        for tid, q in topics.items():
            f.write(f'{tid} (Doc_ID {header_label}):  query = "{q}"\n')
            scores = all_scores[tid][model_key]
            for docid, sc in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:10]:
                f.write(f'{docid} {sc}\n')
            f.write('\n')
    print(f'Wrote {out_path}')

write_appendix('b1', 'BM25_Score',    'Appendix_A_Baseline1_Top10.txt')
write_appendix('b2', 'JM_Score',      'Appendix_B_Baseline2_Top10.txt')
write_appendix('mc', 'ModelC_Score',  'Appendix_C_ModelC_Top10.txt')

Wrote /Users/niceenb/Documents/647/Assignment2/Appendix_A_Baseline1_Top10.txt
Wrote /Users/niceenb/Documents/647/Assignment2/Appendix_B_Baseline2_Top10.txt
Wrote /Users/niceenb/Documents/647/Assignment2/Appendix_C_ModelC_Top10.txt


## 12. User manual (Task 3 deliverable - copy into the report)

**Required Python packages**
```
numpy pandas stemming
```
(install via `pip install numpy pandas stemming`). All other libraries used (`os`, `re`, `math`, `glob`, `string`, `statistics`) are part of the Python standard library. `nltk`, `sklearn` and `matplotlib` are **allowed** by the assignment outline but are not required by this pipeline. `stemming` is the Porter2 stemmer used in the Week 10 workshop solution and is therefore allowed under the "you can re-use workshop solutions" clause.

**Folder layout assumed by the notebook**
```
Assignments/A2/
    IFN647_Assignment2.ipynb       <-- this notebook
    Topics.txt
    Doc_Collection/
        Dataset101/  *.xml
        ...
        Dataset150/
    Relevant_Judgements/
        Dataset101.txt  ...  Dataset150.txt
    ModelOutputs/                  <-- created automatically; will contain
        Baseline1_R###_Ranking.dat
        Baseline2_R###_Ranking.dat
        ModelC_R###_Ranking.dat
```

**How to run** - open the notebook in Jupyter / PyCharm and *Run All Cells*. The pipeline will:
1. parse the 50 topics from `Topics.txt`;
2. parse every XML document in `Doc_Collection/Dataset###/`;
3. compute BM25, JM and Model_C scores for every (topic, document) pair;
4. write 150 ranking files into `ModelOutputs/`;
5. compute AP, P@10 and DCG@10 per topic and per model;
6. perform paired t-tests against the two baselines;
7. write `Table1_AP.csv`, `Table2_P10.csv`, `Table3_DCG10.csv`, `ModelC_Validation.csv`, `Significance_Tests.csv` and `Appendix_*.txt` for the report.

## 13. Notes for the final report (sections to author)

This notebook produces every numerical result the report needs. The remaining work for the report (`Asm2_final_report_template-26.docx`) is to author the discussion sections:

- **Design (Tasks 1 & 2)** - paste the BM25, JM and Model_C pseudocode (Sections 4-6 of this notebook) and explain the parameters.
- **Implementation (Task 3)** - describe `BowDoc`, `BowColl`, `parse_rcv_coll` and the three model functions; include a screenshot of one ranking file from `ModelOutputs/`.
- **Evaluation (Task 4)** - paste `Table1_AP.csv`, `Table2_P10.csv`, `Table3_DCG10.csv` and `ModelC_Validation.csv`. Discuss the validation grid and why the selected parameters were chosen.
- **Significance (Task 5)** - paste `Significance_Tests.csv`. Discuss whether Model_C is significantly better than the baselines on each metric.
- **Application & ethics (Task 6)** - describe a realistic scenario (e.g. an internal news-archive search engine for a journalism agency), discuss ethical issues (bias in the underlying RCV1 collection, privacy of click logs if PRF is moved online, transparency of the learnt expansion weights), propose extensions (BERT embeddings, learning-to-rank with multiple features, LLM-based query expansion) and matching mitigation strategies.
- **Appendices** - paste the three `Appendix_*.txt` files.